# 阶段 5：RL 公式因子挖掘报告

本 Notebook 是阶段 5 的可视化工作台。它不会启动 PPO 训练，只读取配置、数据和训练产物；训练前展示 pre-flight 检查，训练后展示最终因子池、训练诊断和验证指标。

## 复现边界

- 当前只服务论文主线：RPN 公式生成、alpha pool、PPO 奖励和验证评估。
- 不接入真实交易、券商 API、模拟盘或自动下单。
- 不运行 XGBoost；最终因子池目标保持在 25-30 个，配置上限为 30。

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

from quant_rl_alpha.reporting import build_rl_factor_report
from quant_rl_alpha.utils.config import load_config
from quant_rl_alpha.utils.paths import project_root

root = project_root()
rl_config = load_config("rl")
alpha_config = load_config("alpha")

def project_path(path):
    path = Path(path)
    return path if path.is_absolute() else root / path

data_paths = {name: project_path(path) for name, path in rl_config["data"].items() if name in {"daily_panel", "labels"}}
output_paths = {name: project_path(path) for name, path in rl_config["outputs"].items()}

## 配置摘要

In [ ]:
config_summary = pd.DataFrame(
    [
        {"item": "device", "value": rl_config["device"]},
        {"item": "train_iterations", "value": rl_config["train_iterations"]},
        {"item": "episodes_per_iteration", "value": rl_config["episodes_per_iteration"]},
        {"item": "ppo_epochs", "value": rl_config["ppo_epochs"]},
        {"item": "target_pool_size", "value": alpha_config["pool_size"]},
        {"item": "min_valid_dates", "value": alpha_config["min_valid_dates"]},
        {"item": "min_valid_ratio", "value": alpha_config["min_valid_ratio"]},
        {"item": "min_stocks_per_date", "value": alpha_config["min_stocks_per_date"]},
    ]
)
display(config_summary)

## PyTorch / CUDA 状态

In [ ]:
try:
    import torch

    torch_status = {
        "torch_version": torch.__version__,
        "cuda_build": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count(),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
    }
except ModuleNotFoundError:
    torch_status = {"torch_version": "not installed", "cuda_available": False}

display(pd.DataFrame([torch_status]))

## 数据切分与标签覆盖

In [ ]:
labels = pd.read_parquet(data_paths["labels"])
labels["date"] = pd.to_datetime(labels["date"]).dt.normalize()
labels["label_end_date"] = pd.to_datetime(labels["label_end_date"]).dt.normalize()

def label_slice(start, end):
    start = pd.Timestamp(start).normalize()
    end = pd.Timestamp(end).normalize()
    return labels[(labels["date"] >= start) & (labels["date"] <= end) & (labels["label_end_date"] <= end)].copy()

train_labels = label_slice(rl_config["data"]["train_start"], rl_config["data"]["train_end"])
validation_labels = label_slice(rl_config["data"]["validation_start"], rl_config["data"]["validation_end"])
split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_labels),
            "dates": train_labels["date"].nunique(),
            "symbols": train_labels["symbol"].nunique(),
            "start": train_labels["date"].min(),
            "end": train_labels["date"].max(),
            "label_end_max": train_labels["label_end_date"].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_labels),
            "dates": validation_labels["date"].nunique(),
            "symbols": validation_labels["symbol"].nunique(),
            "start": validation_labels["date"].min(),
            "end": validation_labels["date"].max(),
            "label_end_max": validation_labels["label_end_date"].max(),
        },
    ]
)
display(split_summary)

In [ ]:
monthly_labels = labels.groupby("date")["future_20d_return"].agg(["count", "mean", "std"])
display(monthly_labels.tail(12))

## 报告生成

In [ ]:
report_result = build_rl_factor_report()
print(f"mode={report_result.mode}")
print(f"pool={report_result.pool_size}/{report_result.target_pool_size}")
print(f"missing={report_result.missing_artifacts}")
print(f"report={report_result.report_path}")

In [ ]:
report_html = Path(report_result.report_path).read_text(encoding="utf-8")
display(HTML(report_html))

## 训练产物状态

In [ ]:
artifact_status = pd.DataFrame(
    [
        {"artifact": name, "path": path, "exists": path.exists()}
        for name, path in output_paths.items()
    ]
)
display(artifact_status)

## 最终因子池

In [ ]:
pool_path = output_paths["pool"]
if pool_path.exists():
    pool = pd.read_parquet(pool_path)
    min_pool_size = min(25, int(alpha_config["pool_size"]))
    status = "OK" if min_pool_size <= len(pool) <= int(alpha_config["pool_size"]) else "CHECK"
    print(f"pool_size={len(pool)}/{alpha_config['pool_size']} status={status}")
    display(pool.sort_values("weight", key=lambda item: item.abs(), ascending=False))
else:
    print("训练产物尚不存在：先查看上方 pre-flight 报告；完成 stage5-train 后刷新本节。")

## 训练诊断

In [ ]:
metrics_path = output_paths["metrics"]
if metrics_path.exists():
    metrics = pd.read_parquet(metrics_path)
    display(metrics.tail(20))
else:
    print("训练指标尚不存在。")

## 验证期表现

In [ ]:
validation_path = output_paths["validation"]
if validation_path.exists():
    validation = pd.read_parquet(validation_path)
    pool_validation = validation[validation["name"] == "__pool__"]
    factor_validation = validation[validation["name"] != "__pool__"]
    display(pool_validation)
    display(factor_validation.sort_values("validation_ic", ascending=False))
else:
    print("验证指标尚不存在。")